# Spline Signum
We synthesize a spline of a specified degree, delay, period, with random spline coefficients. We display this spline in blue, with blue stems and rings that highlight the samples at the integers, and red stems and rings that highlight the samples at the boundaries of one period. We then extract from this random spline the list of its pieces of constant sign, which we overlay as thick sepia lines and markers. Finally, we print a verbose description of the sign pieces.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Persistent spline
f = sk.PeriodicSpline1D()
f.spline_coeff[0] = rng.standard_normal()

# Plot
def update_plot (
    period = 6,
    degree = 1,
    delay = 0.0
):
    global f # Spline

    # Update of the period
    c = f.spline_coeff
    if f.period < period:
        c = np.append(c, rng.standard_normal(period - f.period))
    else:
        c = c[ : period]
    f = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree, delay = delay)
    # Plot the spline
    (fig, ax) = plt.subplots()
    f.plot((fig, ax), plotpoints = 301, knot_marker = " ")
    # Adjust the display to always show the signum
    (ymin, ymax) = ax.get_ylim()
    ax.set_ylim(bottom = min(ymin, -1.15), top = max(ymax, 1.15))

    # Sign pieces
    pieces = f.piecewise_sgn()
    # Plot each piece independently
    for piece in pieces.pieces:
        lb = piece.domain.infimum # Lower bound of the domain
        ub = piece.domain.supremum # Upper bound of the domain
        if not piece.domain.isleftopen:
            ax.plot(
                lb,
                piece.item,
                marker = "o",
                markerfacecolor = "C5",
                markeredgecolor = "C5",
                markersize = 7.0
            )        
        if not piece.domain.isrightopen:
            ax.plot(
                ub,
                piece.item,
                marker = "o",
                markerfacecolor = "C5",
                markeredgecolor = "C5",
                markersize = 7.0
            )
        if lb == ub:
            continue
        ax.plot(
            [lb, ub],
            [piece.item, piece.item],
            "-C5",
            linewidth = 3.0
        )

    # Show the plot
    plt.show()

    # Verbose description
    print("---")
    for piece in pieces.pieces:
        print(piece)
        print()

widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay)
)